# 05 (Colab) — Arms 2 and 3 on the test set

GPU half of the frozen test run. Produces `arm2_test_predictions.csv` and
`arm3_test_predictions.csv`; `05_results.ipynb` on the CPU side does Arm 1 and all
aggregation. **Nothing here selects anything** — every hyperparameter is read from
`config.HYPERPARAMETERS`, frozen in notebooks 02–04 on validation alone.

Order matters: run `05_results.ipynb` §3 **first**, so `artefacts/arm1_test_predictions.csv`
exists. §4 below rebuilds the Arm 1 test shortlist on this runtime and asserts it reproduces
that file item for item before a single prompt is sent. That guard is the direct response to
the 2026-08-23 incident, where a missing document directory silently degraded the frozen
QLAD index to QLA and invalidated an entire run.

**The test CSV in `results_colab_inputs.zip` has its `answer` column stripped before upload**,
so test-side answers never reach this runtime at all.

Runtime: T4. Arm 2 retrains from scratch (~20–30 min at the frozen 15 epochs) rather than
loading a Drive checkpoint, so this notebook reproduces for anyone with the archive.

## 0. Runtime setup

In [ ]:
!git clone https://github.com/L1N-z/amlh-proj.git
%cd amlh-proj
!pip install -e ".[bert]" -q

In [ ]:
import sys
sys.path.insert(0, "/content/amlh-proj/src")

In [ ]:
import time
import zipfile
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from amlh import arm1_experiments as ae
from amlh import arm2_bert as ab
from amlh import arm3_llm as a3
from amlh import evaluate, features, results
from amlh.config import ARTEFACTS_DIR, HYPERPARAMETERS, PROJECT_ROOT, SEED, set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'}")

### 0.1 Upload the input archive

Built on the workstation with `python scripts/make_results_colab_inputs.py`. The repo clone
carries neither `data/` nor `artefacts/` — `.gitignore` excludes both — so everything has to
arrive through this zip.

In [ ]:
from google.colab import files

if not Path("results_colab_inputs.zip").exists():
    files.upload()  # select results_colab_inputs.zip

!unzip -o -q results_colab_inputs.zip -d .
print(f"unzipped into {Path.cwd()}")

### 0.2 Input integrity check

Every assertion here corresponds to a way a previous run went wrong, or could. The document
coverage check is the one that matters most: the `D` component of the frozen `QLAD` variant
is the NHS document text, and its absence is what silently degraded the index last time.

In [ ]:
train = pd.read_csv(PROJECT_ROOT / "data" / "patient_qa_classification_train.csv")
assert train["disease"].nunique() == 906, train["disease"].nunique()

test_raw = pd.read_csv(PROJECT_ROOT / "data" / "patient_qa_classification_test.csv")
assert "answer" not in test_raw.columns, "test CSV carries answers -- rebuild the archive"
assert len(test_raw) == 200, len(test_raw)
print(f"test CSV columns: {list(test_raw.columns)}  <- no `answer`")

coverage = features.doc_coverage(train["disease"].unique())
assert coverage["missing"] == [], f"{len(coverage['missing'])} classes have no NHS document"
print(f"NHS documents: {coverage['n_found']}/{coverage['n_total']} classes covered")

for name in ("split_fit.csv", "split_val.csv", "arm1_test_predictions.csv", "arm2_val_metrics.csv"):
    assert (ARTEFACTS_DIR / name).is_file(), f"missing artefacts/{name} -- rebuild the archive"
print("artefacts present: split_fit.csv, split_val.csv, arm1_test_predictions.csv, arm2_val_metrics.csv")

## 1. Freeze check

Same check as the CPU notebook. A `None` here would mean a choice was never made and this
run would be inventing one.

In [ ]:
set_seed()

FROZEN_FIELDS = [
    "index_variant", "index_scheme", "ngram_range", "min_df", "sublinear_tf", "k_neighbors",
    "bert_model_name", "max_length", "learning_rate", "batch_size", "num_epochs",
    "shortlist_k", "llm_temperature", "prompt_mode", "arm3_model_name", "arm3_max_new_tokens",
]
frozen = {name: getattr(HYPERPARAMETERS, name) for name in FROZEN_FIELDS}
unset = [name for name, value in frozen.items() if value is None]
assert not unset, f"hyperparameters still unfrozen: {unset}"

for name, value in frozen.items():
    print(f"  {name:22s} = {value!r}")
print(f"\nall {len(frozen)} hyperparameters frozen | SEED = {SEED}")

## 2. Load the frozen inputs

`fit` is `split_fit` — the 8,691-row training half, not the full training set. CLAUDE.md
fixes the tested model as the validated model, so there is no refit on fit + val.

In [ ]:
fit = pd.read_csv(ARTEFACTS_DIR / "split_fit.csv")
val = pd.read_csv(ARTEFACTS_DIR / "split_val.csv")
test = pd.read_csv(PROJECT_ROOT / "data" / "patient_qa_classification_test.csv")

results.assert_no_answer_column(test)
print(f"fit  = {len(fit):5d} rows, {fit.disease.nunique()} classes")
print(f"val  = {len(val):5d} rows, {val.disease.nunique()} classes")
print(f"test = {len(test):5d} rows, {test.disease.nunique()} classes")

## 3. Arm 2 — Bio_ClinicalBERT on test

Retrained rather than loaded from a checkpoint, so this notebook stands alone. The call
below is identical to the one `03_arm2_bert_colab.ipynb` used for its final refit — same
seed, same data, same frozen epoch count — and that refit reproduced the ablation's
per-item predictions exactly (`refit_agreement = 1.0` in `arm2_val_metrics.csv`).

`val` is passed because `train_model` scores each epoch on it; that is the validated
training procedure and involves no test data. §3b re-checks the validation accuracy the
retrain lands on, which is the evidence that this is the same model that was selected.

In [ ]:
set_seed()

label_to_id, id_to_label = ab.encode_labels(train)
print(f"label space: {len(label_to_id)} classes")

truncation = ab.truncation_rate(test["question"].tolist(),
                                AutoTokenizer.from_pretrained(HYPERPARAMETERS.bert_model_name),
                                HYPERPARAMETERS.max_length)
print(f"test truncation rate at max_length={HYPERPARAMETERS.max_length}: {truncation:.4f}")

In [ ]:
set_seed()
start = time.perf_counter()

state_dict, history, _ = ab.train_model(
    fit, val, label_to_id, HYPERPARAMETERS.bert_model_name,
    lr=HYPERPARAMETERS.learning_rate,
    batch_size=HYPERPARAMETERS.batch_size,
    epochs=HYPERPARAMETERS.num_epochs,
    max_length=HYPERPARAMETERS.max_length,
    seed=SEED,
)
print(f"trained {HYPERPARAMETERS.num_epochs} epochs in {(time.perf_counter() - start) / 60:.1f} min")
history.tail()

### 3b. Confirm the retrain reproduces the validated model

If this validation accuracy does not match the 0.850 recorded in `arm2_val_metrics.csv`, the
retrain is not the selected model and the test predictions below would describe a different
system. Printed rather than asserted: cuDNN kernel selection is not bit-deterministic, so a
small disagreement is a hardware artefact. A large one is a logic error — stop and
investigate rather than proceeding.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    HYPERPARAMETERS.bert_model_name, num_labels=len(label_to_id)
)
model.load_state_dict(state_dict)
model = model.to(device)
tokeniser = AutoTokenizer.from_pretrained(HYPERPARAMETERS.bert_model_name)

val_ranked = ab.predict_ranked(
    model, tokeniser, val["question"].tolist(), id_to_label,
    max_length=HYPERPARAMETERS.max_length, batch_size=HYPERPARAMETERS.batch_size,
    device=device, top_k=None,
)
val_accuracy = evaluate.score_ranked(val_ranked, val["disease"].tolist())["accuracy"]

recorded = pd.read_csv(ARTEFACTS_DIR / "arm2_val_metrics.csv")["accuracy"].iloc[0]
print(f"retrain val accuracy: {val_accuracy:.4f} | recorded in arm2_val_metrics.csv: {recorded:.4f}")
print(f"delta: {abs(val_accuracy - recorded):.4f}")
if abs(val_accuracy - recorded) > 0.02:
    print("WARNING: retrain diverged from the selected model by more than 2pp. Investigate before continuing.")

### 3c. Arm 2 test predictions

`top_k=None` ranks all 906 labels. Arm 1 ranks every class, so truncating here would make
Arm 2's MRR an MRR@k and the two arms' columns non-comparable.

In [ ]:
test_ranked = ab.predict_ranked(
    model, tokeniser, test["question"].tolist(), id_to_label,
    max_length=HYPERPARAMETERS.max_length, batch_size=HYPERPARAMETERS.batch_size,
    device=device, top_k=None,
)

arm2_test = pd.DataFrame({"question": test["question"], "gold": test["disease"]})
arm2_test["pred"] = [r[0] for r in test_ranked]
for i in range(results.SHORTLIST_DEPTH):
    arm2_test[f"top_{i + 1}"] = [r[i] if i < len(r) else None for r in test_ranked]

arm2_test.to_csv(ARTEFACTS_DIR / "arm2_test_predictions.csv", index=False)
print(f"Arm 2 test accuracy: {(arm2_test['pred'] == arm2_test['gold']).mean():.4f}")
arm2_test[["question", "gold", "pred"]].head()

In [ ]:
# Free the encoder before loading the generator -- both will not fit in 16 GB together.
del model
torch.cuda.empty_cache()
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Arm 1 test shortlist, and proof it is the frozen one

`require_doc_coverage` raises rather than degrading when the `D` component is missing. The
reproduction assert is the second guard: rank 1 is a sharp fingerprint of the index that
produced it, so if this runtime's shortlist disagrees with the CPU-side
`arm1_test_predictions.csv` on even one item, the index is not the frozen one and the run
stops here — before any GPU time is spent on prompts.

In [ ]:
set_seed()

features.require_doc_coverage(fit.disease.unique())

shortlists, top_sim = a3.build_shortlist_ranking(
    fit, test, HYPERPARAMETERS, depth=a3.shortlist_k(HYPERPARAMETERS)
)
results.assert_reproduces_arm1_test(shortlists)

gold_in_shortlist = sum(g in s for g, s in zip(test["disease"], shortlists)) / len(test)
print(f"shortlist depth {len(shortlists[0])} | gold present in {gold_in_shortlist:.3f} of shortlists")

## 5. Arm 3 — frozen prompt condition on the frozen generator

`prompt_mode` and `arm3_model_name` come from `config.py`, selected in `04_arm3_llm.ipynb`
by the pre-registered two-stage rule. Only the selected cell runs here; the other five cells
of the validation grid were reported for transparency and are not systems.

In [ ]:
set_seed()

mode = HYPERPARAMETERS.prompt_mode
model_name = HYPERPARAMETERS.arm3_model_name
print(f"frozen condition: {mode} | frozen generator: {model_name}")

_, _, pipe = a3.load_generator(model_name, device=device)
examples = a3.build_examples(fit, a3.n_shots(HYPERPARAMETERS), seed=SEED)

In [ ]:
set_seed()
start = time.perf_counter()

arm3_test, arm3_metrics, _ = a3.run_condition(
    fit, test, HYPERPARAMETERS,
    mode=mode,
    pipe=pipe,
    examples=examples,
    model_name=model_name,
    shortlist_rankings=shortlists,
    top_sim=top_sim,
)
arm3_test.to_csv(ARTEFACTS_DIR / "arm3_test_predictions.csv", index=False)

print(f"ran in {time.perf_counter() - start:.1f}s")
for key, value in arm3_metrics.items():
    print(f"  {key}: {value}")

### 5b. What the LLM did to the shortlist

The decomposition that made the validation result interpretable, repeated on test. On
validation: 118 items kept at rank 1 (no change), 32 fallbacks (inert by construction), and
50 active re-ranks that scored 0.120 where the retriever scored 0.640 — an intervention
precision of 6/50.

In [ ]:
decomposition = results.rerank_decomposition(arm3_test)
decomposition

## 6. Zip the artefacts for download

Copy both CSVs into the workstation's `artefacts/`, then re-run `05_results.ipynb` — it will
pick up all three arms and produce the full comparison, McNemar matrix and error analysis.

In [ ]:
zip_path = Path("results_colab_outputs.zip")
members = ["arm2_test_predictions.csv", "arm3_test_predictions.csv"]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in members:
        path = ARTEFACTS_DIR / name
        assert path.is_file(), f"{name} was never written"
        zf.write(path, arcname=f"artefacts/{name}")

print(f"wrote {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB)")
for name in zipfile.ZipFile(zip_path).namelist():
    print(f"  {name}")

files.download(str(zip_path))